<a href="https://colab.research.google.com/github/kimdonggyu2008/Personal_Study/blob/main/cosyvoice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/코딩공부/project_folder

/content/drive/MyDrive/코딩공부/project_folder


In [3]:
!git clone https://github.com/FunAudioLLM/CosyVoice.git

Cloning into 'CosyVoice'...
remote: Enumerating objects: 2276, done.
remote: Counting objects: 100% (681/681), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 2276 (delta 571), reused 496 (delta 496), pack-reused 1595 (from 2)
Receiving objects: 100% (2276/2276), 1.60 MiB | 7.49 MiB/s, done.
Resolving deltas: 100% (1420/1420), done.


### 코드 분석

In [ ]:
import torch
from torch import nn, sin, pow
from torch.nn import Parameters

import math
from typing import Tuple
from torch import nn

In [ ]:
class Swish(torch.nn.Module):
  def forward(self,x: torch.Tensor)-> torch.Tensor:
    return x*torch.sigmoid(x)

class Snake(nn.Module):
  def __init__(self,in_feature,alpha=1.0 alpha_trainable=True,alpha_logscale=False):

    super(Snake,self).__init__()
    self.in_features=in_features

    self.alpha_logscale=alpha_logscale
    if self.alpha_logscale:
      self.alpha=Parameter(torch.zeros(in_features)*alpha)

    else:
      self.alpha=Parameter(torch.ones(in_features)*alpha)

    self.alpha.requires_grad=alpha_trainable

    self.no_div_by_zero=0.000000001


  def forward(self,x):
    alpha=self.alpha.unsqueeze(0).unsqueeze(-1)
    if self.alpha_logscale:
      alpha=torch.exp(alpha)
    x=x+(1.0/(alpha+self.no_div_by_zero))*pow(sin(x*alpha),2)

    return x

In [ ]:
# 멀티헤드 어텐션 구현

class MultiHeadAttention(nn.Module):
  def __init__(self,
               n_head:int,
               n_feat:int,
               dropout_rate:float,
               key_bias:bool=True):

    super().__init__()
    assert n_feat%n_head==0

    self.d_k=n_feat//n_head
    self.h=n_head
    self.linear_q=nn.Linear(n_feat,n_feat)
    self.linear_k=nn.Linear(n_feat,n_feat,bias=key_bias)
    self.linear_v=nn.Linear(n_feat,n_feat)
    self.linear_out=nn.Linear(n_feat,n_feat)
    self.dropout=nn.Dropout(p=dropout_rate)

    def forward_qkv(
        self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
      n_batch=query.size(0)
      q=self.linear_q(query).view(n_batch,-1,self.h,self.d_k)
      k=self.linear_k(key).view(n_batch,-1,self.h,self.d_k)
      v=self.linear_v(value).view(n_batch,-1,self.h,self.d_k)

      q=q.transpose(1,2)
      k=k.transpose(1,2)
      v=v.transpose(1,2)

      return q,k,v


    def forward_attention(
        self,
        value=torch.Tensor,
        scores: torch.Tensor,
        mask:torch.Tensor=torch.ones((0,0,0),dtype=torch.bool)
    )->torch.Tensor:
      n_batch=value.size(0)
      if mask.size(2)>0:
        mask=mask.unsqueeze(1).eq(0)
        mask=mask[:,:,:,:scores.size(-1)]
        scores=scores.masked_fill(mask,-float('inf'))
        attn=torch.softmax(scores,dim=-1).masked_fill(mask,0.0)

      else:
        attn=torch.softmax(scores,dim=-1)

      p_attn=self.dropout(attn)
      x=torch.matmul(p_attn,value)
      x=(x.transpose(1,2).contiguous().view(n_batch,-1,self.h*self.d_k))

      return self.linear_out(x)

  def forward(
      self,
      query:torch.Tensor,
      key:torch.Tensor,
      value:torch.Tensor,
      mask:torch.Tensor=torch.ones((0,0,0),dtype=torch.bool),
      pos_emb:torch.Tensor=torch.empty(0),
      cache:torch.Tensor=torch.zeros((0,0,0,0))
  )->Tuple[torch.Tensor,torch.Tensor]:

    q,k,v=self.forward_qkv(query,key,value)
    if cache.size(0)>0:
      key_cache, value_cache=torch.split(cache,
                                         cache.size(-1)//2,
                                         dim=-1)
      k=torch.cat([key_cache,k],dim=2)
      v=torch.cat([value_cache,v],dim=2)
    new_cache=torch.cat((k,v),dim=-1)

    scores=torch.matmul(q,k.transpose(-2,-1))/math.sqrt(self.d_k)
    return self.forward_attention(v,scores,mask),new_cache





In [ ]:
class RelPositionMultiHeadAttention(MultiHeadttention):
  def __init__(self,
               n_head:int,
               n_feat:int,
               dropout_rate:float,
               key_bias:bool=True):
    super().__init__(n_head,n_feat,dropout_rate,key_bias)
    self.linear_pos=nn.Linear(n_feat,n_feat,bias=False)

    self.pos_bias_u=nn.Parameter(torch.Tensor(self.h,self.d_k))
    self.pos_bias_v=nn.Parameter(torch.Tensor(self.h,self.d_k))
    torch.nn.init.xavier_uniform_(self.pos_bias_u)
    torch.nn.init.xavier_uniform_(self.pos_bias_v)

  def rel_shift(self,x:torch.Tensor)->torch.Tensor:
    zero_pad=torch.zeros((x.size()[0],x.size()[1],x.size()[2],1),
                         device=x.device,
                         dtype=x.dtype)
    x_padded=torch.cat([zero_pad,x],dim=-1)
    x_padded=x_padded.view(x.size()[0],
                           x.size()[1],
                           x.size(3)+1,x.size(2))
    x=x_padded[:,:,1:].view_as(x)[
        :,:,:,: x.size(-1)//2+1
    ]
    return x

  def forward(
      self,
      query: torch.Tensor,
      key:torch.Tensor,
      value:torch.Tensor,
      mask:torch.Tensor=torch.ones((0,0,0),dtype=torch.bool),
      pos_emb:torch.Tensor=torch.empty(0),
      cache:torch.Tensor=torch.zeros((0,0,0,0))
  )->Tuple[torch.Tensor,torch.Tensor]

    q,k,v=self.forward_qkv(query,key,value)
    q=q.transpose(1,2)

    if cache.size(0)>0:
      key_cache,value_cache=torch.split(cache,cache.size(-1)//2,dim=-1)
      k=torch.cat([key_cache,k],dim=2)
      v=torch.cat([value_cache,v],dim=2)

    new_cache=torch.cat((k,v),dim=-1)

    n_bach_pos=pos_emb.size(0)
    p=self.linear_pos(pos_emb).view(n_batch_pos,-1,self.h,self.d_k)

    q_with_bias_u=(q+self.pos_bias_u).transpose(1,2)
    q_with_bias_v=(q+self.pos_bias_v).transpose(1,2)

    matrix_ac=torch.matmul(q_with_bias_u,k.transpose(-2,-1))

    matrix_bd=torch.matmul(q_with_bias_v,p.transpose(-2,-1))

    if matrix_ac.shape!=matrix_bd.shape:
      matrix_bd=self.rel_shift(matrix_bd)

    scores=(matrix_ac+matrix_bd)/math.sqrt(self.d_k)

    return self.forward_attention(v,scores,mask),new_cache

In [ ]:
class ConvolutionModule(nn.Module):

  def __init__(self,
               channels:int,
               kernel_size:int=15,
               activation:nn.Module=nn.ReLU(),
               norm:str="batch_norm",
               casual:bool=False,
               bias:bool=True):
    super().__init__()

    self.pointwise_conv1=nn.Conv1d(
        channels,
        2*channels,
        kernel_size=1,
        stride=1,
        padding=0,
        bias=bias,
    )

    if casual:
      padding=0
      self.lorder=kernel_size-1
    else:
      assert(kernel_size-1)%2==0
      padding=(kernel_size-1)//2
      self.lorder=0

    self.depthwise_conv=Conv1d(
        channels,
        channels,
        kernel_size,
        stride=1,
        padding=padding,
        groups=channels,
        bias=bias,
    )

    assert norm in ['batch_norm','layer_norm']
    if norm=="batch_norm":
      self.use_layer_norm=False
      self.norm=nn.BatchNorm1d(channels)
    else:
      self.use_layer_norm=True
      self.norm=nn.LayerNorm(channels)

    self.pointwise_conv2=nn.Conv1d(
        channels,
        channels,
        kernel_size=1,
        stride=1,
        padding=0,
        bias=bias,
    )
    self.activation=activation


  def forward(
      self,
      x:torch.Tensor,
      mask_pad:torch.Tensor=torch.ones((0,0,0),dtype=torch.bool),
      cache:torch.Tensor=torch.zeros((0,0,0)),
  )->Tuple[torch.Tensor,torch.Tensor]:

    x=x.transpose(1,2)

    if mask_pad.size(2)>0:
      x.masked_fill_(-mask_pad,0.0)

    if self.lorder>0:
      if cache.size(2)==0:
        x=nn.functional.pad(x,(self.lorder,0),'constant',0.0)
      else:
        assert cache.size(0)==x.size(0)
        assert cache.size(1)==x.size(1)
        x=torch.cat((cache,x),dim=2)
      assert (x.size(2)>self.lorder)
      new_cache=x[:,:,-self.lorder:]
    else:
      new_cache=torch.zeros((0,0,0),dtype=x.dtype,device=x.device)


    x=self.pointwise_conv1(x)
    x=nn.functional.glu(x,dim=1)

    x=self.depthwise_conv(x)
    if self.use_layer_norm:
      x=x.transpose(1,2)
    x=self.activation(self.norm(x))
    if self.use_layer_norm:
      x=x.transpose(1,2)
      x=self.pointwise_conv2(x)

    if mask_pad.size(2)>0:
      x.masked_fill_(-mask_pad,0.0)

    return x.transpose(1,2),new_cache

In [ ]:
class DecoderLayer(nn.Module):

  def __init__(
      self,
      size_attn:nn.Module,
      src_attn:Optional[nn.Module],
      feed_forward:nn.Module,
      dropout_rate:float,
      normalize_before:bool=True,
  ):

  super().__init__()
  self.size=size
  self.self_attn=self_attn
  self.src_attn=src_attn
  self.feed_forward=feed_forward
  self.norm1=nn.LayerNorm(size,eps=1e-5)
  self.norm2=nn.LayerNorm(size,eps=1e-5)
  self.norm3=nn.LayerNorm(size,eps=1e-5)
  self.dropout=nn.Dropout(dropout_rate)
  self.normalize_before=normalize_before


  def forward(
      self
      tgt:torch.Tensor,
      tgt_mask:torch.Tensor,
      memory:torch.Tensor,
      memory_mask:torch.Tensor,
      cache:Optional[torch.Tensor]=None,
  ) -> Tuple[torch.Tensor,torch.Tensor,torch.Tensor,torch.Tensor]:

    residual=tgt
    if self.normalize_before:
      tgt=self.norm(tgt)

    if cache is None:
      tgt_q=tgt
      tgt_q_mask=tgt_mask

    else:
      assert cache.shape==(
          tgt.shape[0],
          tgt.shape[1]-1,
          self.size,
      ),"{cache.shape} == {(tgt.shape[0], tgt.shape[1] - 1, self.size)}"
      tgt_q = tgt[:, -1:, :]
      residual=residual[:,-1:,:]
      tgt_q_mask=tgt_mask[:,-1:,:]

    x=residual+self.dropout(
        self.self_attn(tgt_q,tgt,tgt,tgt_q_mask)[0])
    if no self.normalize_before:
      x=self.norm1(x)

    if self.src_attn is not None:
      residual=x
      if self.normalize_before:
        x=self.norm2(x)
      x=residual+self.dropout(
          self.src_attn(x,memory,memory,memory_mask)[0])
      if not self.normalize_before:
        x=self.norm2(x)

    residual=x
    if self.normalize_before:
      x=self.norm3(x)
    x=residual+self.dropout(self.feed_forward(x))
    if not self.normalize_before:
      x=self.norm3(x)

    if cache is not None:
      x=torch.cat([cache,x],dim=1)

    return x, tgt_mask,memory,memory_mask

In [ ]:
class PositionalEncoding(torch.nn.Module):
  def __init__(self,
               d_model:int,
               dropout_rate:float,
               max_len:int=5000,
               reverse:bool=False):

    super().__init__()
    self.d_model=d_model
    self.xscale=math.sqrt(self.d_model)
    self.dropout=torch.nn.Dropout(p=dropout_rate)
    self.max_len=max_len

    self.pe=torch.zeros(self.max_len,self.d_model)
    position=torch.arange(0,self.max_len,dtype=torch.float32).unsqueeze(1)

    div_term=torch.exp(torch.arange(0,self.d_model,2,dtype=torch.float32)* -(math.log(10000.0)/self.d_model))
    self.pe[:,0::2]=torch.sin(position*div_term)
    self.pe[:,1::2]=torch.cos(position*div_term)
    self.pe=self.pe.unsqueeze(0)


  def forward(self,
              x:torch.Tensor,
              offset:Union[int,torch.Tensor]=0) -> Tuple[torch.Tensor,torch.Tensor]:

    self.pe=self.pe.to(x.device)
    pos_emb=self.position_encoding(offset,x.size(1),Fasle)
    x=x*self.xscale+pos_emb
    return self.dropout(x),self.dropout(pos_emb)

  def position_encoding(self,
                        offset:Union[int,torch.Tensor],
                        size:int,
                        apply_dropout:bool=True) -> torch.Tensor:

    if isinstance(offset,int):
      assert offset+size<=self.max_len
      pos_emb=self.pe[:,offset:offset+size]

    elif isinstance(ofset,torch.Tensor) and offset.dim()==0:
      assert offset+size<=self.max_len
      pos_emb=self.pe[:,offset:offset+size]

    else:
      assert torch.max(offset)+size<=self.max_len
      index=offset.unsqueeze(1)+torch.arange(0,size).to(offset.device)
      flag=index>0
      flag=index*flag
      pos_emb=F.embedding(index,self.pe[0])

    if apply_dropout:
      pos_emb=self.dropout(pos_emb)
    return pos_emb




In [ ]:
class RelPositionalEncoding(PositionalEncoding):
  def __init__(self,d_model,int,dropout_rate: float,max_len:int=5000):
    super().__init__(d_model,dropout_rate,max_len,reverse=True)

  def forward(self,
              x:torch.Tensor,
              offset:Union[int,torch.Tensor]=0)->Tuple[torch.Tensor,torch.Tensor]:
    self.pe=self.pe.to(x.device)
    x=x*self.xscale
    pos_emb=self.position_encoding(offset,x.size(1),False)
    return self.dropout(x),self.dropout(pos_emb)

In [ ]:
class WhisperPositionalEncoding(PositionalEncoding):
  def __init__(self,d_model:int, dropout_rate:float,max_len:int=5000):
    super().__init__(d_model,dropout_rate,max_len)
    self.xscale=1.0
    log_timescale_increment=np.log(10000)/(d_model//2-1)
    inv_timescales=torch.exp(-log_timescale_increment*torch.arange(d_model//2))
    scaled_time=torch.arange(max_len)[:,np.newaxis]*inv_timescales[np.nexaxis,:]
    pe=torch.cat([torch.sin(scaled_time),torch.cos(scaled_tiem)],dim=1)
    delattr(self,"pe")
    self.register_buffer("pe",pe.unsqueeze(0))

In [ ]:
class LearnablePositionalEncoding(PositionalEncoding):
  def __init__(self,d_model: int, dropout_rate:float, max_len:int=448):
    super().__init__(d_model,dropout_rate,max_len)
    self.pe=torch.nn.Parameter(torch.empty(1,max_len,d_model))
    self.xscale=1.0

class NoPositionalEncoding(torch.nn.Module):
  def __init__(self,d_model:int, dropout_rate:float):
    super().__init__()
    self.d_model=d_model
    self.dropout=torch.nn.Dropout(p=dropout_rate)

  def forward(self,
              x:torch.Tensor,
              offset:Union[int,torch.Tensor]=0) -> Tuple[torch.Tensor,torch.Tensor]:

    pos_emb=torch.zeros(1,x.size(1),self.d_model).to(x.device)
    return self.dropout(x),pos_emb

  def position_encoding(self,offset:Union[int,torch.Tensor],
                        size:int)->torch.Tensor:
    return torch.zeros(1,size,self.d_model)


class EspnetRelPositionalEncoding(torch.nn.Module):
  def __init__(self,d_model:int,dropout_rate:float,max_len:int=5000):
    super(EspnetRelPositionalEncoding,self).__init__()
    self.d_model=d_model
    self.xscale=math.sqrt(self.d_model)
    self.dropout=torch.nn.Dropout(p=dropout_rate)
    self.pe=None
    self.extend_pe(torch.tensor(0.0).expand(1,max_len))

  def extend_pe(self,x:torch.Tensor):
    if self.pe is not None:
      if self.pe.size(1)>=x.size(1)*2-1:
        if self.pe.dtype!=x.dtype or self.pe.device!=x.device:
          self.pe=self.pe.to(dtype=x.dtype,device=x.device)
        return

    pe_positive=torch.zeros(x.size(1),self.d_model)
    pe_negative=torch.zeros(x.size(1),self.d_model)
    position=torch.arange(0,x.size(1),dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(
            torch.arange(0, self.d_model, 2, dtype=torch.float32)
            * -(math.log(10000.0) / self.d_model)
        )

    pe_positive[:, 0::2] = torch.sin(position * div_term)
    pe_positive[:, 1::2] = torch.cos(position * div_term)
    pe_negative[:, 0::2] = torch.sin(-1 * position * div_term)
    pe_negative[:, 1::2] = torch.cos(-1 * position * div_term)

    pe_positive=torch.flip(pe_positive,[0]).unsqueeze(0)
    pe_negative=pe_negative.unsqueeze(0)
    pe=torch.cat([pe_positive,pe_neagtive],dim=1)
    self.pe=pe.to(device=x.device,dtype=x.dtype)

  def forward(self,x:torch.Tensor,offset:Union[int,torch.Tensor]=0) -> Tuple[torch.Tensor,torch.Tensor]:
    self.extend_pe(x)
    x=x*self.xscale
    pos_emb=self.position_encoding(size=x.size(1),offset=offset)
    return self.dropout(x),self.dropout(pos_emb)

  def position_encoding(self,
                      offset: Union[int, torch.Tensor],
                      size: int) -> torch.Tensor:
    """ For getting encoding in a streaming fashion

    Attention!!!!!
    we apply dropout only once at the whole utterance level in a none
    streaming way, but will call this function several times with
    increasing input size in a streaming scenario, so the dropout will
    be applied several times.

    Args:
        offset (int or torch.tensor): start offset
        size (int): required size of position encoding

    Returns:
        torch.Tensor: Corresponding encoding
    """
    # How to subscript a Union type:
    #   https://github.com/pytorch/pytorch/issues/69434
    if isinstance(offset, int):
      pos_emb = self.pe[
          :,
          self.pe.size(1) // 2 - size - offset + 1: self.pe.size(1) // 2 + size + offset,
      ]
    elif isinstance(offset, torch.Tensor):
      pos_emb = self.pe[
          :,
          self.pe.size(1) // 2 - size - offset + 1: self.pe.size(1) // 2 + size + offset,
      ]
    return pos_emb

In [2]:
class TransformerEncoderLayer(nn.Module):
  def __init__(
      self,
      size:int,
      self_attn:torch.nn.Module,
      feed_forward:torch.nn.Module,
      feed_forward:torch.nn.Module,
      dropout_rate:float,
      normalize_before:bool=True,
  ):
    super().__init__()
    self.self_attn=self_attn
    self.feed_forward=feed_forward
    self.norm1=nn.LayerNorm(size,eps=1e-5)
    self.norm2=nn.LayerNorm(size,eps=1e-5)
    self.dropout=nn.Dropout(dropout_rate)
    self.size=size
    self.normalize_before=normalize_before

  def forward(
      self,
      x:torch.Tensor,
      mask:torch.Tensor,
      pos_emb=torch.Tensor,
      mask_pad:torch.Tensor=torch.ones((0,0,0),dtype=torch.bool),
      att_cache:torch.Tensor=torch.zeros((0,0,0,0)),
      cnn_cache:torch.Tensor=torch.zeros((0,0,0,0)),
      )->Tuple[torch.Tensor, torch.Tensor,torch.Tensor,torch.Tensor]:

    residual=x
    if slef.normalize_before:
      x=self.norm1(x)
    x_att,new_att_cache=self.self_attn(x,x,x,mask,pos_emb=pos_emb,cache=att_cache)
    x=residual+self.dropout(x_att)
    if not self.normalize_before:
      x=self.norm1(x)

    residual=x
    if self.normalize_before:
      x=self.norm2(x)
    x=residual+self.dropout(self.feed_forward(x))
    if not self.normalize_before:
      x=self.norm2(x)

    fake_cnn_cache=torch.zeros((0,0,0),dtype=x.dtype,device=x.device)
    return x,mask,new_att_cache,fake_cnn_cache


class ConformerEncoderLayer(nn.Module):
 def __init__(
     self,
     size:int,
     self_attn:torch.nn.Module,
     feed_forward:Optional[nn.Module]=None,
     feed_forward_macaron:Optional[nn.Module]=None,
     conv_module:Optional[nn.Module]=None,
     dropout_rate:float=0.1,
     normalize_before:bool=True,
    ):

    super().__init__()
    self.self_attn=self.attn
    self.feed_forward=feed_forward
    self.feed_forward_macaron=feed_forward_macaron
    self.conv_module=conv_module
    self.norm_ff=nn.LayerNorm(size,eps=1e-12)
    self.norm_mha=nn.LayerNorm(size,eps=1e-12)
    if feed_forward is not None:
      self.norm_ff_macaron=nn.LayerNorm(size,eps=1e-12)
      self.ff_scale=0.5
    else:
      self.ff_scale=1.0

    if self.conv_module is not None:
      self.norm_conv=nn.LayerNorm(size,eps=1e-12)
      self.norm_final=nn.LayerNorm(
          size, eps=1e-12)
    self.dropout=nn.Dropout(dropout_rate)
    self.size=size
    self.normalize_before=normalize_before


  def forward(
        self,
        x: torch.Tensor,
        mask: torch.Tensor,
        pos_emb: torch.Tensor,
        mask_pad: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
        att_cache: torch.Tensor = torch.zeros((0, 0, 0, 0)),
        cnn_cache: torch.Tensor = torch.zeros((0, 0, 0, 0)),
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:


    # whether to use macaron style
    if self.feed_forward_macaron is not None:
      residual = x
      if self.normalize_before:
        x = self.norm_ff_macaron(x)
      x = residual + self.ff_scale * self.dropout(
        self.feed_forward_macaron(x))
      if not self.normalize_before:
        x = self.norm_ff_macaron(x)

    # multi-headed self-attention module
    residual = x
    if self.normalize_before:
      x = self.norm_mha(x)
    x_att, new_att_cache = self.self_attn(x, x, x, mask, pos_emb,
                                          att_cache)
    x = residual + self.dropout(x_att)
    if not self.normalize_before:
      x = self.norm_mha(x)

    # convolution module
    # Fake new cnn cache here, and then change it in conv_module
    new_cnn_cache = torch.zeros((0, 0, 0), dtype=x.dtype, device=x.device)
    if self.conv_module is not None:
      residual = x
      if self.normalize_before:
        x = self.norm_conv(x)
      x, new_cnn_cache = self.conv_module(x, mask_pad, cnn_cache)
      x = residual + self.dropout(x)

      if not self.normalize_before:
        x = self.norm_conv(x)

    # feed forward module
    residual = x
    if self.normalize_before:
      x = self.norm_ff(x)

    x = residual + self.ff_scale * self.dropout(self.feed_forward(x))
    if not self.normalize_before:
      x = self.norm_ff(x)

    if self.conv_module is not None:
      x = self.norm_final(x)

    return x, mask, new_att_cache, new_cnn_cache

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 83)

In [ ]:
class LabelSmoothingLoss(nn.Module):
  def __init__(self,
               size:int,
               padding_idx:int,
               smoothing:float,
               normalize_length:bool=False):
    super(LabelSmoothingLoss,self).__init__()
    self.criterion=nn.KLDivLoss(reduction="none")
    self.padding_idx=padding_idx
    self.confidence=1.0-smoothing
    self.smoothing=smoothing
    self.size=size
    self.normalize_length=normalize_length

  def forward(self,x:torch.Tensor,target:torch.Tensor)->torch.Tensor:
    assert x.size(2)==self.size
    batch_size=x.size(0)
    x=x.view(-1,self.size)
    target=target.view(-1)
    true_dist=torch.zeros_like(x)
    true_dixt.fill_(self.smoothing/(self.size-1))
    ignore=target==self.padding_idx
    total=len(target)==self.padding_idx
    target=target.masked_fill(ignore,0)
    true_dist.scatter_(1,target.unsqueeze(1),self.confidence)
    kl=self.criterion(torch.log_softmax(x,dim=1),true_dist)
    denom=total if self.normalize_length else batch_size
    return kl.masked_fill(ignore.unsqueeze(1),0).sum/denom